# Diagnóstico Bronze — twitch_top_200_games_data

Tabla analizada: `steam_game_intelligence.bronze.twitch_top_200_games_data`. Se comprueban granularidad mensual, rangos de métricas, rankings duplicados y valores ausentes. Se respetan los nombres de columnas del esquema, incluidos los nombres con mayúsculas.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y cobertura

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT Game) AS distinct_games,
  COUNT(DISTINCT Year) AS distinct_years,
  MIN(Year) AS min_year,
  MAX(Year) AS max_year,
  MIN(Month) AS min_month,
  MAX(Month) AS max_month,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

## 2. Nulos por columna

In [0]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(`Rank` IS NULL) AS Rank,
    COUNT_IF(Game IS NULL) AS Game,
    COUNT_IF(Month IS NULL) AS Month,
    COUNT_IF(Year IS NULL) AS Year,
    COUNT_IF(Hours_watched IS NULL) AS Hours_watched,
    COUNT_IF(Hours_streamed IS NULL) AS Hours_streamed,
    COUNT_IF(Peak_viewers IS NULL) AS Peak_viewers,
    COUNT_IF(Peak_channels IS NULL) AS Peak_channels,
    COUNT_IF(Streamers IS NULL) AS Streamers,
    COUNT_IF(Avg_viewers IS NULL) AS Avg_viewers,
    COUNT_IF(Avg_channels IS NULL) AS Avg_channels,
    COUNT_IF(Avg_viewer_ratio IS NULL) AS Avg_viewer_ratio,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.twitch_top_200_games_data
) UNPIVOT (null_count FOR column_name IN (
  `Rank`, Game, Month, Year, Hours_watched, Hours_streamed, Peak_viewers,
  Peak_channels, Streamers, Avg_viewers, Avg_channels, Avg_viewer_ratio,
  _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Validación de año y mes

In [0]:
%sql
SELECT
  COUNT_IF(Year IS NOT NULL AND (Year < 2016 OR Year > 2023)) AS year_outside_expected_range,
  COUNT_IF(Month IS NOT NULL AND (Month < 1 OR Month > 12)) AS invalid_month,
  COUNT_IF(`Rank` IS NOT NULL AND `Rank` < 1) AS invalid_rank,
  COUNT_IF(`Rank` IS NOT NULL AND `Rank` > 200) AS rank_above_top200,
  COUNT_IF(Game IS NOT NULL AND TRIM(Game) = '') AS blank_game,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

## 4. Duplicados de ranking dentro del mes

In [0]:
%sql
SELECT Year, Month, `Rank`, COUNT(*) AS row_count,
       COUNT(DISTINCT Game) AS distinct_games
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
GROUP BY Year, Month, `Rank`
HAVING COUNT(*) > 1
ORDER BY Year, Month, `Rank`
LIMIT 200;

## 5. Juegos repetidos en un mismo mes

In [0]:
%sql
SELECT Year, Month, LOWER(TRIM(Game)) AS normalized_game,
       COUNT(*) AS row_count,
       COUNT(DISTINCT `Rank`) AS distinct_ranks,
       MIN(`Rank`) AS best_rank,
       MAX(`Rank`) AS worst_rank
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
WHERE Game IS NOT NULL AND TRIM(Game) <> ''
GROUP BY Year, Month, LOWER(TRIM(Game))
HAVING COUNT(*) > 1
ORDER BY row_count DESC
LIMIT 200;

## 6. Métricas negativas o inconsistentes

In [0]:
%sql
SELECT
  COUNT_IF(Hours_watched < 0) AS negative_hours_watched,
  COUNT_IF(Hours_streamed < 0) AS negative_hours_streamed,
  COUNT_IF(Peak_viewers < 0) AS negative_peak_viewers,
  COUNT_IF(Peak_channels < 0) AS negative_peak_channels,
  COUNT_IF(Streamers < 0) AS negative_streamers,
  COUNT_IF(Avg_viewers < 0) AS negative_avg_viewers,
  COUNT_IF(Avg_channels < 0) AS negative_avg_channels,
  COUNT_IF(Avg_viewer_ratio < 0) AS negative_avg_viewer_ratio
FROM steam_game_intelligence.bronze.twitch_top_200_games_data;

In [0]:
%sql
SELECT *
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
WHERE `Rank` < 1 OR `Rank` > 200
   OR Month < 1 OR Month > 12
   OR Year < 2016 OR Year > 2023
   OR Hours_watched < 0 OR Hours_streamed < 0
   OR Peak_viewers < 0 OR Peak_channels < 0
   OR Streamers < 0 OR Avg_viewers < 0 OR Avg_channels < 0
   OR Avg_viewer_ratio < 0
   OR Game IS NULL OR TRIM(Game) = ''
LIMIT 200;

## 7. Cobertura por año y mes

In [0]:
%sql
SELECT Year, Month,
       COUNT(*) AS rows_in_month,
       COUNT(DISTINCT Game) AS distinct_games,
       MIN(`Rank`) AS best_rank,
       MAX(`Rank`) AS worst_rank
FROM steam_game_intelligence.bronze.twitch_top_200_games_data
GROUP BY Year, Month
ORDER BY Year, Month;

In [0]:
%python
from datetime import datetime
import os

table_name = 'steam_game_intelligence.bronze.twitch_top_200_games_data'
notebook_name = '04_diagnostico_twitch_top200'
queries = [('Consulta de celda SQL 1', 'SELECT\n  COUNT(*) AS total_rows,\n  COUNT(DISTINCT Game) AS distinct_games,\n  COUNT(DISTINCT Year) AS distinct_years,\n  MIN(Year) AS min_year,\n  MAX(Year) AS max_year,\n  MIN(Month) AS min_month,\n  MAX(Month) AS max_month,\n  MIN(_ingested_at) AS first_ingested_at,\n  MAX(_ingested_at) AS last_ingested_at\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data;'), ('Consulta de celda SQL 2', 'SELECT column_name, null_count\nFROM (\n  SELECT\n    COUNT_IF(`Rank` IS NULL) AS Rank,\n    COUNT_IF(Game IS NULL) AS Game,\n    COUNT_IF(Month IS NULL) AS Month,\n    COUNT_IF(Year IS NULL) AS Year,\n    COUNT_IF(Hours_watched IS NULL) AS Hours_watched,\n    COUNT_IF(Hours_streamed IS NULL) AS Hours_streamed,\n    COUNT_IF(Peak_viewers IS NULL) AS Peak_viewers,\n    COUNT_IF(Peak_channels IS NULL) AS Peak_channels,\n    COUNT_IF(Streamers IS NULL) AS Streamers,\n    COUNT_IF(Avg_viewers IS NULL) AS Avg_viewers,\n    COUNT_IF(Avg_channels IS NULL) AS Avg_channels,\n    COUNT_IF(Avg_viewer_ratio IS NULL) AS Avg_viewer_ratio,\n    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,\n    COUNT_IF(_source IS NULL) AS _source\n  FROM steam_game_intelligence.bronze.twitch_top_200_games_data\n) UNPIVOT (null_count FOR column_name IN (\n  `Rank`, Game, Month, Year, Hours_watched, Hours_streamed, Peak_viewers,\n  Peak_channels, Streamers, Avg_viewers, Avg_channels, Avg_viewer_ratio,\n  _ingested_at, _source\n))\nWHERE null_count > 0\nORDER BY null_count DESC;'), ('Consulta de celda SQL 3', "SELECT\n  COUNT_IF(Year IS NOT NULL AND (Year < 2016 OR Year > 2023)) AS year_outside_expected_range,\n  COUNT_IF(Month IS NOT NULL AND (Month < 1 OR Month > 12)) AS invalid_month,\n  COUNT_IF(`Rank` IS NOT NULL AND `Rank` < 1) AS invalid_rank,\n  COUNT_IF(`Rank` IS NOT NULL AND `Rank` > 200) AS rank_above_top200,\n  COUNT_IF(Game IS NOT NULL AND TRIM(Game) = '') AS blank_game,\n  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data;"), ('Consulta de celda SQL 4', 'SELECT Year, Month, `Rank`, COUNT(*) AS row_count,\n       COUNT(DISTINCT Game) AS distinct_games\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data\nGROUP BY Year, Month, `Rank`\nHAVING COUNT(*) > 1\nORDER BY Year, Month, `Rank`\nLIMIT 200;'), ('Consulta de celda SQL 5', "SELECT Year, Month, LOWER(TRIM(Game)) AS normalized_game,\n       COUNT(*) AS row_count,\n       COUNT(DISTINCT `Rank`) AS distinct_ranks,\n       MIN(`Rank`) AS best_rank,\n       MAX(`Rank`) AS worst_rank\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data\nWHERE Game IS NOT NULL AND TRIM(Game) <> ''\nGROUP BY Year, Month, LOWER(TRIM(Game))\nHAVING COUNT(*) > 1\nORDER BY row_count DESC\nLIMIT 200;"), ('Consulta de celda SQL 6', 'SELECT\n  COUNT_IF(Hours_watched < 0) AS negative_hours_watched,\n  COUNT_IF(Hours_streamed < 0) AS negative_hours_streamed,\n  COUNT_IF(Peak_viewers < 0) AS negative_peak_viewers,\n  COUNT_IF(Peak_channels < 0) AS negative_peak_channels,\n  COUNT_IF(Streamers < 0) AS negative_streamers,\n  COUNT_IF(Avg_viewers < 0) AS negative_avg_viewers,\n  COUNT_IF(Avg_channels < 0) AS negative_avg_channels,\n  COUNT_IF(Avg_viewer_ratio < 0) AS negative_avg_viewer_ratio\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data;'), ('Consulta de celda SQL 7', "SELECT *\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data\nWHERE `Rank` < 1 OR `Rank` > 200\n   OR Month < 1 OR Month > 12\n   OR Year < 2016 OR Year > 2023\n   OR Hours_watched < 0 OR Hours_streamed < 0\n   OR Peak_viewers < 0 OR Peak_channels < 0\n   OR Streamers < 0 OR Avg_viewers < 0 OR Avg_channels < 0\n   OR Avg_viewer_ratio < 0\n   OR Game IS NULL OR TRIM(Game) = ''\nLIMIT 200;"), ('Consulta de celda SQL 8', 'SELECT Year, Month,\n       COUNT(*) AS rows_in_month,\n       COUNT(DISTINCT Game) AS distinct_games,\n       MIN(`Rank`) AS best_rank,\n       MAX(`Rank`) AS worst_rank\nFROM steam_game_intelligence.bronze.twitch_top_200_games_data\nGROUP BY Year, Month\nORDER BY Year, Month;')]
queries.append(("Muestra de estructura (una fila)", f"SELECT * FROM {table_name} LIMIT 1"))

def markdown_value(value):
    if value is None:
        return "NULL"
    return str(value).replace("|", "\\|").replace("\r", " ").replace("\n", " ")

report = [
    f"# Resultados del diagn?stico ? {notebook_name}",
    "",
    f"- **Notebook:** `{notebook_name}`",
    f"- **Ejecutado:** `{datetime.now().astimezone().isoformat()}` (zona horaria del driver)",
    "- **Estado:** `completed`",
    "- **Objetivo:** conservar los resultados efectivos de las validaciones SQL del notebook para su revisi?n local.",
    "",
    "## Tablas inspeccionadas",
    f"- `{table_name}`",
    "",
    "## Consultas y resultados",
]
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(["", f"### {title}", f"_Filas exportadas: {len(rows)}; m?ximo aplicado: 200._", ""])
    if not rows:
        report.append("No se detectaron registros que incumplan esta validaci?n o la consulta no devolvi? filas.")
        continue
    columns = list(rows[0].keys())
    report.append("| " + " | ".join(columns) + " |")
    report.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        report.append("| " + " | ".join(markdown_value(row.get(column)) for column in columns) + " |")
report.extend(["", "## Hallazgos y conclusiones", "Los hallazgos deben interpretarse a partir de los conteos y muestras exportados; esta celda no inventa conclusiones ni marca como aprobada una validaci?n no ejecutada.", ""])

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/notebooks/data_validations"
if marker not in context_path:
    raise RuntimeError(f"No se pudo determinar la ra?z del repositorio desde la ruta del notebook: {context_path}")
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
output_dir = os.path.join(workspace_repo_path, "notebooks", "data_validations", "result_outputs")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, f"resultados_{notebook_name}.md")
temp_path = output_path + ".tmp"
with open(temp_path, "w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(report))
os.replace(temp_path, output_path)
print(f"Markdown generado: {output_path}; consultas exportadas: {len(queries)}")
